# 1회차 타이타닉 코드 (결측치 처리 + 원핫인코딩 전처리만 사용)

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

### 여기서 시각화, 통계 분석등 자유롭게

# 3) 사용 컬럼만 선택
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

### 종속변수, 독립변수 선택도 목표에 맞게 수정 가능

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 이건 그대로~ (실무에서는 굳이 이 과정을 안해도 될 수 있음)

# 5) 결측치 처리 (train 기준으로 값 계산)
age_median = X_train["Age"].median()
emb_mode = X_train["Embarked"].mode()[0]

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"]  = X_test["Age"].fillna(age_median)

X_train["Embarked"] = X_train["Embarked"].fillna(emb_mode)
X_test["Embarked"]  = X_test["Embarked"].fillna(emb_mode)

# 6) 범주형 원-핫 인코딩 (train 기준 컬럼에 맞춰 test 정렬)
X_train_enc = pd.get_dummies(X_train, columns=["Sex", "Embarked"], drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=["Sex", "Embarked"], drop_first=True)

# train의 컬럼 구조에 맞춰 test를 재정렬 (없으면 0으로 채움)
#-> 지금 단계에서는 꼭 안해도 되는데, 만약 train에는 있는 컬럼이 test에는 없다면?
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# 8) GridSearchCV (DecisionTree, max_dept)
base_clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [None, 2, 3, 4, 5, 7, 10]
}

grid = GridSearchCV(
    estimator=base_clf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_enc, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 9) test 평가
best_clf = grid.best_estimator_
pred = best_clf.predict(X_test_enc)

print("\nTest Accuracy:", accuracy_score(y_test, pred))


결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'max_depth': 5}
Best CV Score: 0.8188909681867429

Test Accuracy: 0.7653631284916201


# 2회차 타이타닉 코드 (스케일링만 간단하게 추가)
- 물론 트리기반 모델은 굳이 스케일링을 하지 않아도 됌
- 하지만, Linear Regression, Logistic Rgression, SVM 등의 모델에서는 필요
- 참고로 스케일링은 수치형 데이터에만 적용가능, 또한 원-핫인코딩 적용된(0과 1로만 이루어진) 컬럼은 굳이 스케일링을 진행하지 않음

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) 사용 컬럼만 선택
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5) 결측치 처리 (train 기준)
age_median = X_train["Age"].median()
emb_mode = X_train["Embarked"].mode()[0]

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"]  = X_test["Age"].fillna(age_median)

X_train["Embarked"] = X_train["Embarked"].fillna(emb_mode)
X_test["Embarked"]  = X_test["Embarked"].fillna(emb_mode)

# 6) 범주형 원-핫 인코딩 + 컬럼 정렬
X_train_enc = pd.get_dummies(X_train, columns=["Sex", "Embarked"], drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=["Sex", "Embarked"], drop_first=True)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# 7) 스케일링 추가 (train에 fit, test는 transform)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

### 이렇게 간단하게 추가할 수 있는데... 한가지 아쉬운 부분이 있다. -> 원-핫인코딩 진행했던 컬럼은 굳이 스케일링을 진행할 필요가 없음

# 8) GridSearchCV (DecisionTree, max_depth)
base_clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [None, 2, 3, 4, 5, 7, 10]
}

grid = GridSearchCV(
    estimator=base_clf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 9) test 평가
best_clf = grid.best_estimator_
pred = best_clf.predict(X_test_scaled)

print("\nTest Accuracy:", accuracy_score(y_test, pred))



결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'max_depth': 5}
Best CV Score: 0.8174825174825175

Test Accuracy: 0.7653631284916201


# 2회차 타이타닉 코드 (수치형, 범주형 각각 전처리하고 나중에 합치기)

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) 사용 컬럼 선택
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5) 결측치 처리 (train 기준)
age_median = X_train["Age"].median()
emb_mode = X_train["Embarked"].mode()[0]

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"]  = X_test["Age"].fillna(age_median)

X_train["Embarked"] = X_train["Embarked"].fillna(emb_mode)
X_test["Embarked"]  = X_test["Embarked"].fillna(emb_mode)



# (A) 수치형 전처리
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num  = scaler.transform(X_test[num_cols])

# scaler를 사용하면 numpy array 형태로 변환이 되어서 ㅜㅜ
# 다시 pandas 기능을 쓰고 싶으면 DataFrame 형태로 변환해야 합니다
# ndarray → DataFrame으로 변환
X_train_num_df = pd.DataFrame(
    X_train_num,
    columns=num_cols,
    index=X_train.index
)

X_test_num_df = pd.DataFrame(
    X_test_num,
    columns=num_cols,
    index=X_test.index
)



# (B) 범주형 원-핫 인코딩
cat_cols = ["Sex", "Embarked"]

X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True)
X_test_cat  = pd.get_dummies(X_test[cat_cols], drop_first=True)
# 컬럼 정렬
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)


# 각각 전처리 다 끝났으니 결합
# (C) 수치형 + 범주형 결합
X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
X_test_final  = pd.concat([X_test_num_df, X_test_cat], axis=1)


# 우리가 아는 돼지국밥
# 8) GridSearchCV
base_clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [None, 2, 3, 4, 5, 7, 10]
}

grid = GridSearchCV(
    estimator=base_clf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_final, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 9) test 평가
best_clf = grid.best_estimator_
pred = best_clf.predict(X_test_final)

print("\nTest Accuracy:", accuracy_score(y_test, pred))



결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'max_depth': 5}
Best CV Score: 0.8174825174825175

Test Accuracy: 0.7653631284916201


# 2회차 타이타닉 코드 (위 코드를 Pipeline으로 정리 하기)

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) 사용 컬럼 선택
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# (A) 컬럼 구분
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]


# (B) 수치형 파이프라인
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# (C) 범주형 파이프라인
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# (D) 전처리 통합
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


# (E) 전체 Pipeline 구성
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])


# 8) GridSearchCV (max_depth만 튜닝)
param_grid = {
    "clf__max_depth": [None, 2, 3, 4, 5, 7, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 9) test 평가
best_model = grid.best_estimator_
pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, pred))



결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'clf__max_depth': 4}
Best CV Score: 0.8174825174825175

Test Accuracy: 0.7932960893854749


# 2회차 타이타닉 코드 (Count Vectorizer까지 추가하고 Pipeline으로 정리 하기)

In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) Name 컬럼 추가
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Name"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# (A) 컬럼 구분
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]
text_col = "Name"


# (B) 수치형 파이프라인
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# (C) 범주형 파이프라인

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# (D) 텍스트 파이프라인 (CountVectorizer)
text_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(min_df=2))
])


# (E) 전처리 통합
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("txt", text_pipeline, text_col)
])


# (F) 전체 Pipeline 구성
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])


# 8) GridSearchCV (max_depth만 튜닝)
param_grid = {
    "clf__max_depth": [None, 2, 3, 4, 5, 7, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# 9) test 평가
best_model = grid.best_estimator_
pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, pred))



결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'clf__max_depth': 10}
Best CV Score: 0.8077218556091795

Test Accuracy: 0.8268156424581006
